# Notebook Overview — Classifier Selection and Tuning

## Purpose

This notebook performs **model selection, comparison, and hyperparameter tuning** for the Digital Image Processing (DIP)–based AI image detection pipeline.

It evaluates multiple candidate classifiers using **stratified cross-validation**, identifies top-performing models, and refines them through **controlled hyperparameter tuning**, preparing final model candidates for independent test evaluation.

---

## Inputs

Required inputs:

* Normalized feature vector CSV file:
  * `metadata/vectors/<train_feature_vectors_normalized_filename>`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* Normalized training feature vectors are loaded
* Dataset structure, metadata, and feature dimensionality are validated
* Feature matrix (`X_train`) and target vector (`y_train`) are constructed
* Class labels are encoded into numeric form
* A diverse set of candidate classifiers is defined
* Stratified 5-fold cross-validation is configured
* Baseline performance is evaluated across all classifiers
* Results are aggregated and ranked based on ROC AUC
* Top-performing classifiers are selected for tuning
* Hyperparameter tuning is performed using `GridSearchCV`
* Tuned model performance is recorded and compared
* Results are saved for downstream use
* Final classifier recommendations are generated
* Processing is deterministic and reproducible

---

## Outputs

**Baseline Classifier Results**  
`metadata/results/<baseline_model_results_filename>`

**Tuned Classifier Results**  
`metadata/results/<tuned_model_results_filename>`

**Best Classifier Configurations**  
`metadata/results/<best_model_config_filename>`

Each results file contains:

* Classifier performance metrics:
  * Accuracy
  * Precision
  * Recall
  * F1 Score
  * ROC AUC

* Cross-validation statistics:
  * Mean values
  * Standard deviation

* Tuned results additionally include:
  * Optimal hyperparameters
  * Best ROC AUC score

---

## Expected Behavior

* All candidate classifiers are evaluated successfully using cross-validation
* Performance metrics are stable across folds
* Top-performing classifiers (typically **RBF SVM and MLP**) are identified
* Hyperparameter tuning improves or refines model performance
* Results are correctly saved to CSV and JSON files
* Selected classifiers are suitable for final independent evaluation
* No test data is used during model selection, preventing data leakage

---
---

### 🔷 Step 1 — Startup: Imports, Configuration, and Verification

- Import required standard and third-party libraries for model training and evaluation
- Configure notebook display behavior using `VERBOSE`
- Optionally suppress warnings for cleaner output
- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define input path for normalized training feature vectors
- Define output paths for classifier comparison results and best model configuration
- Create results output directory if it does not exist
- Verify that required normalized training data is available
- Optionally display configuration details when `VERBOSE=True`

---

In [1]:
# ============================================
# Step 1: Startup — Imports, Configuration, and Verification
# ============================================

# -------------------------------------------------
# Standard library imports
# -------------------------------------------------
import os
import sys
import json
import warnings
from pathlib import Path

# -------------------------------------------------
# Third-party imports
# -------------------------------------------------
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

import time
from itertools import product

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Suppress warnings for cleaner output (optional)
# -------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    VECTORS_METADATA_DIR,
    RESULTS_METADATA_DIR,
    TRAIN_NORMALIZED_FILENAME,
    CLASSIFIER_COMPARISON_BASELINE_FILENAME,
    CLASSIFIER_COMPARISON_TUNED_FILENAME,
    BEST_MODEL_CONFIG_FILENAME,
    NUM_FEATURES,
    RANDOM_SEED,
)

# -------------------------------------------------
# Define input and output file paths
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = Path(VECTORS_METADATA_DIR) / TRAIN_NORMALIZED_FILENAME

BASELINE_RESULTS_CSV = Path(RESULTS_METADATA_DIR) / CLASSIFIER_COMPARISON_BASELINE_FILENAME
TUNED_RESULTS_CSV = Path(RESULTS_METADATA_DIR) / CLASSIFIER_COMPARISON_TUNED_FILENAME
BEST_CLASSIFIER_JSON = Path(RESULTS_METADATA_DIR) / BEST_MODEL_CONFIG_FILENAME

# -------------------------------------------------
# Ensure results directory exists
# -------------------------------------------------
Path(RESULTS_METADATA_DIR).mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input file
# -------------------------------------------------
print("Verifying required normalized training CSV file...\n")

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    raise FileNotFoundError(
        f"Missing required file: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}"
    )

print("Required normalized training CSV file is present.")

# -------------------------------------------------
# Optional verbose display of configuration
# -------------------------------------------------
if VERBOSE:
    print(f"Input file:         {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
    print(f"Output directory:   {RESULTS_METADATA_DIR}")
    print(f"Expected features:  {NUM_FEATURES}")
    print(f"Random seed:        {RANDOM_SEED}")
    print("Libraries imported successfully.")



Cloning repository...
Verifying required normalized training CSV file...

Required normalized training CSV file is present.
Input file:         /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv
Output directory:   /content/dip-ai-image-detection/metadata/results
Expected features:  26
Random seed:        42
Libraries imported successfully.


### 🔷 Step 2 — Load Normalized Training Data

- Display path to normalized training dataset
- Load normalized training feature vector CSV file
- Display dataset shape for verification
- Optionally preview first few rows of the dataset
- Optionally display column names when `VERBOSE=True`

---

In [2]:
# ============================================
# Step 2: Load Normalized Training Data
# ============================================

# -------------------------------------------------
# Display input file path
# -------------------------------------------------
print(f"Loading training data from: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")

# -------------------------------------------------
# Load normalized training dataset
# -------------------------------------------------
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

print("\nTraining dataset loaded successfully.")
print(f"Shape: {df_train.shape}")

# -------------------------------------------------
# Optional preview of dataset
# -------------------------------------------------
if VERBOSE:
    print("\nFirst 5 rows:")
    try:
        display(df_train.head())  # Jupyter/Colab
    except:
        print(df_train.head())    # Fallback

# -------------------------------------------------
# Optional display of column names
# -------------------------------------------------
if VERBOSE:
    print("\nColumn names:")
    for col in df_train.columns:
        print(col)



Loading training data from: /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv

Training dataset loaded successfully.
Shape: (14400, 30)

First 5 rows:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Noise Residual Energy,Low Frequency Energy Ratio,Mid Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.071297,-0.189219,0.111086,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,-0.330229,-0.264290,0.292101,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.851489,-0.544033,0.442499,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.188882,0.518813,-0.552523,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.281104,0.607047,-0.600533,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939



Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
Mid Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std


### 🔷 Step 3 — Validate Training Data

- Check for missing values in the dataset
- Display total number of missing values
- Optionally display class distribution when `VERBOSE=True`
- Verify presence of required metadata columns
- Identify feature columns by excluding metadata
- Validate that feature count matches expected DIP feature dimension
- Confirm dataset structure is correct before model training

---

In [3]:
# ============================================
# Step 3: Validate Training Data
# ============================================

print("Performing sanity checks...\n")

# -------------------------------------------------
# Check for missing values
# -------------------------------------------------
missing_values = df_train.isnull().sum().sum()
print(f"Total missing values: {missing_values}")

if missing_values == 0:
    print("No missing values detected.")
else:
    print("WARNING: Missing values detected.")

# -------------------------------------------------
# Display class distribution (optional)
# -------------------------------------------------
class_counts = df_train["class_label"].value_counts()

if VERBOSE:
    print("\nClass distribution:")
    print(class_counts)

# -------------------------------------------------
# Verify required metadata columns
# -------------------------------------------------
expected_metadata_cols = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

print("\nChecking metadata columns...")
for col in expected_metadata_cols:
    if col in df_train.columns:
        if VERBOSE:
            print(f"OK: {col}")
    else:
        raise ValueError(f"Missing required column: {col}")

# -------------------------------------------------
# Identify feature columns
# -------------------------------------------------
feature_cols = [
    col for col in df_train.columns
    if col not in expected_metadata_cols
]

print(f"\nNumber of feature columns: {len(feature_cols)}")

# -------------------------------------------------
# Validate feature count
# -------------------------------------------------
EXPECTED_FEATURE_COUNT = NUM_FEATURES

if len(feature_cols) == EXPECTED_FEATURE_COUNT:
    print("Feature count is correct.")
else:
    raise ValueError(
        f"Expected {EXPECTED_FEATURE_COUNT} features, "
        f"found {len(feature_cols)}."
    )

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nValidation complete.")



Performing sanity checks...

Total missing values: 0
No missing values detected.

Class distribution:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Checking metadata columns...
OK: filename
OK: class_label
OK: source_dataset
OK: subset

Number of feature columns: 26
Feature count is correct.

Validation complete.


### 🔷 Step 4 — Prepare Feature Matrix and Target Vector

- Extract feature matrix `X_train` from feature columns
- Extract target vector `y_train` from class labels
- Display shapes of feature matrix and target vector
- Encode class labels into numeric values using `LabelEncoder`
- Optionally display label encoding mapping when `VERBOSE=True`
- Optionally display encoded class distribution
- Confirm data is ready for model training

---

In [4]:
# ============================================
# Step 4: Prepare X_train and y_train
# ============================================

print("Preparing feature matrix X and target vector y...\n")

# -------------------------------------------------
# Build feature matrix (X) and target vector (y)
# -------------------------------------------------
X_train = df_train[feature_cols].values
y_train = df_train["class_label"].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# -------------------------------------------------
# Encode class labels to numeric values
# -------------------------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# -------------------------------------------------
# Optional verbose display of label mapping
# -------------------------------------------------
if VERBOSE:
    print("\nLabel encoding mapping:")
    for i, label in enumerate(label_encoder.classes_):
        print(f"{label} -> {i}")

# -------------------------------------------------
# Optional verbose display of encoded distribution
# -------------------------------------------------
if VERBOSE:
    print("\nEncoded class distribution:")
    unique, counts = np.unique(y_train_encoded, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"Class {u}: {c}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nPreparation complete.")



Preparing feature matrix X and target vector y...

X_train shape: (14400, 26)
y_train shape: (14400,)

Label encoding mapping:
ai -> 0
rl -> 1

Encoded class distribution:
Class 0: 7200
Class 1: 7200

Preparation complete.


### 🔷 Step 5 — Define Candidate Classifiers

- Define stratified 5-fold cross-validation strategy
- Define candidate baseline classifiers for model comparison
- Include linear, nonlinear, ensemble, distance-based, probabilistic, and neural-network classifiers
- Define evaluation metrics for classifier comparison
- Optionally display classifier names, scoring metrics, and cross-validation settings when `VERBOSE=True`
- Confirm classifier comparison setup is complete

---

In [5]:
# ============================================
# Step 5: Define Candidate Classifiers
# ============================================

print("Defining candidate classifiers and scoring metrics...\n")

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

# -------------------------------------------------
# Define candidate classifiers
# -------------------------------------------------
candidate_classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    ),

    "Linear SVM": SVC(
        kernel="linear",
        probability=True,
        random_state=RANDOM_SEED
    ),

    "RBF SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_SEED
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Gaussian NB": GaussianNB(),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=RANDOM_SEED
    )
}

# -------------------------------------------------
# Define scoring metrics
# -------------------------------------------------
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# -------------------------------------------------
# Optional verbose display of classifier setup
# -------------------------------------------------
if VERBOSE:
    print("Candidate classifiers:")
    for name in candidate_classifiers.keys():
        print(f" - {name}")

    print("\nScoring metrics:")
    for metric in scoring_metrics.keys():
        print(f" - {metric}")

    print("\nCross-validation strategy:")
    print(" - StratifiedKFold")
    print(" - n_splits = 5")
    print(" - shuffle = True")
    print(f" - random_state = {RANDOM_SEED}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nClassifier setup complete.")



Defining candidate classifiers and scoring metrics...

Candidate classifiers:
 - Logistic Regression
 - Linear SVM
 - RBF SVM
 - Random Forest
 - Extra Trees
 - Gradient Boosting
 - KNN
 - Gaussian NB
 - MLP

Scoring metrics:
 - accuracy
 - precision
 - recall
 - f1
 - roc_auc

Cross-validation strategy:
 - StratifiedKFold
 - n_splits = 5
 - shuffle = True
 - random_state = 42

Classifier setup complete.


### 🔷 Step 6 — Run Baseline Classifier Comparison

- Perform cross-validation for each candidate classifier
- Use stratified 5-fold cross-validation for fair evaluation
- Evaluate classifiers using multiple performance metrics:
  - Accuracy
  - Precision
  - Recall
  - F1 score
  - ROC AUC
- Record mean and standard deviation for each metric
- Measure execution time for each classifier
- Store results for later comparison and analysis
- Note that this step may take several minutes depending on hardware

---

In [6]:
# ============================================
# Step 6: Run Baseline Classifier Comparison
# ============================================

print("Running baseline classifier comparison...\n")

# -------------------------------------------------
# NOTE:
# This step performs cross-validation across multiple classifiers
# and may take several minutes depending on hardware.
#
# Performance can be improved by:
# - Using a higher-memory environment (e.g., Colab High-RAM)
# - Running on a machine with more CPU cores
#
# Parallel execution is enabled via n_jobs=-1
# -------------------------------------------------

import time

# -------------------------------------------------
# Initialize results storage
# -------------------------------------------------
baseline_results = []
total_models = len(candidate_classifiers)

# -------------------------------------------------
# Evaluate each classifier using cross-validation
# -------------------------------------------------
for i, (name, clf) in enumerate(candidate_classifiers.items(), 1):
    print(f"[{i}/{total_models}] Evaluating: {name}")

    start_time = time.time()

    cv_results = cross_validate(
        estimator=clf,
        X=X_train,
        y=y_train_encoded,
        cv=cv_strategy,
        scoring=scoring_metrics,
        return_train_score=False,
        n_jobs=-1
    )

    elapsed = time.time() - start_time

    # -------------------------------------------------
    # Aggregate cross-validation results
    # -------------------------------------------------
    result = {
        "classifier": name,
        "accuracy_mean": np.mean(cv_results["test_accuracy"]),
        "accuracy_std": np.std(cv_results["test_accuracy"]),
        "precision_mean": np.mean(cv_results["test_precision"]),
        "precision_std": np.std(cv_results["test_precision"]),
        "recall_mean": np.mean(cv_results["test_recall"]),
        "recall_std": np.std(cv_results["test_recall"]),
        "f1_mean": np.mean(cv_results["test_f1"]),
        "f1_std": np.std(cv_results["test_f1"]),
        "roc_auc_mean": np.mean(cv_results["test_roc_auc"]),
        "roc_auc_std": np.std(cv_results["test_roc_auc"]),
        "fit_time_sec": elapsed
    }

    baseline_results.append(result)

    print(f"Completed: {name} (time: {elapsed:.2f} sec)\n")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("Baseline classifier comparison complete.")



Running baseline classifier comparison...

[1/9] Evaluating: Logistic Regression
Completed: Logistic Regression (time: 2.25 sec)

[2/9] Evaluating: Linear SVM
Completed: Linear SVM (time: 77.07 sec)

[3/9] Evaluating: RBF SVM
Completed: RBF SVM (time: 54.34 sec)

[4/9] Evaluating: Random Forest
Completed: Random Forest (time: 9.10 sec)

[5/9] Evaluating: Extra Trees
Completed: Extra Trees (time: 2.38 sec)

[6/9] Evaluating: Gradient Boosting
Completed: Gradient Boosting (time: 24.09 sec)

[7/9] Evaluating: KNN
Completed: KNN (time: 0.91 sec)

[8/9] Evaluating: Gaussian NB
Completed: Gaussian NB (time: 0.17 sec)

[9/9] Evaluating: MLP
Completed: MLP (time: 25.48 sec)

Baseline classifier comparison complete.


### 🔷 Step 7 — Compile and Rank Results

- Convert baseline classifier results into a DataFrame
- Sort classifiers by mean ROC AUC score
- Optionally display the full ranked comparison table when `VERBOSE=True`
- Select the top-ranked classifiers for summary display
- Report ROC AUC, F1 score, and accuracy for the top classifiers
- Confirm baseline ranking is complete

---

In [7]:
## ============================================
# Step 7: Compile and Rank Results
# ============================================

print("Compiling and ranking baseline classifier results...\n")

# -------------------------------------------------
# Convert baseline results to DataFrame
# -------------------------------------------------
df_baseline_results = pd.DataFrame(baseline_results)

# -------------------------------------------------
# Sort results by primary metric (ROC-AUC)
# -------------------------------------------------
df_baseline_results = df_baseline_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Optional verbose display of full results
# -------------------------------------------------
if VERBOSE:
    print("Baseline classifier comparison (sorted by ROC-AUC):\n")

    try:
        display(df_baseline_results)
    except:
        print(df_baseline_results)

# -------------------------------------------------
# Display top-ranked classifiers
# -------------------------------------------------
TOP_K = 3

print(f"\nTop {TOP_K} classifiers based on ROC-AUC:\n")

for i in range(min(TOP_K, len(df_baseline_results))):
    row = df_baseline_results.iloc[i]
    print(
        f"{i+1}. {row['classifier']} | "
        f"ROC-AUC: {row['roc_auc_mean']:.4f} | "
        f"F1: {row['f1_mean']:.4f} | "
        f"Accuracy: {row['accuracy_mean']:.4f}"
    )

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nRanking complete.")



Compiling and ranking baseline classifier results...

Baseline classifier comparison (sorted by ROC-AUC):



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,fit_time_sec
0,RBF SVM,0.777917,0.006806,0.772321,0.006300,0.788194,0.009160,0.780163,0.007091,0.860830,0.005220,54.343101
1,MLP,0.778958,0.005700,0.771693,0.007137,0.792500,0.011959,0.781885,0.006210,0.859406,0.005201,25.482660
2,Random Forest,0.764583,0.005437,0.769011,0.008333,0.756528,0.004272,0.762688,0.004479,0.846517,0.005605,9.095402
3,Extra Trees,0.759444,0.004947,0.759028,0.009083,0.760556,0.006264,0.759727,0.003480,0.844014,0.005678,2.375716
4,Gradient Boosting,0.749375,0.006673,0.746176,0.007236,0.755972,0.010755,0.750998,0.007090,0.831207,0.007006,24.087116
5,Linear SVM,0.727292,0.005725,0.712298,0.005585,0.762639,0.007472,0.736596,0.005705,0.799648,0.009323,77.066296
6,Logistic Regression,0.725903,0.004471,0.714860,0.004671,0.751667,0.009050,0.732769,0.005059,0.799193,0.009304,2.254093
7,KNN,0.727431,0.002001,0.718159,0.002143,0.748750,0.010019,0.733089,0.003969,0.793156,0.004763,0.911816
8,Gaussian NB,0.666458,0.004100,0.662448,0.006244,0.679167,0.011761,0.670610,0.004802,0.710773,0.006359,0.167037



Top 3 classifiers based on ROC-AUC:

1. RBF SVM | ROC-AUC: 0.8608 | F1: 0.7802 | Accuracy: 0.7779
2. MLP | ROC-AUC: 0.8594 | F1: 0.7819 | Accuracy: 0.7790
3. Random Forest | ROC-AUC: 0.8465 | F1: 0.7627 | Accuracy: 0.7646

Ranking complete.


### 🔷 Step 8 — Tune Top Classifiers

- Define parameter grids for selected classifiers
- Select top-performing classifiers from baseline results
- Perform hyperparameter tuning using `GridSearchCV`
- Use stratified cross-validation with ROC AUC as the optimization metric
- Evaluate all parameter combinations for each classifier
- Record best ROC AUC score and corresponding hyperparameters
- Measure and record training time for each model
- Store tuning results for comparison and final selection
- Confirm tuning process is complete

---

In [8]:
# ============================================
# Step 8: Tune Top Classifiers
# ============================================

print("Tuning top classifiers...\n")

# -------------------------------------------------
# Initialize storage for tuned results
# -------------------------------------------------
tuned_results = []

# -------------------------------------------------
# Define parameter grids (controlled search space)
# -------------------------------------------------
param_grids = {
    "RBF SVM": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.1, 0.01]
    },

    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64, 32)],
        "alpha": [0.0001, 0.001],
        "learning_rate_init": [0.001, 0.0005]
    },

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20]
    }
}

# -------------------------------------------------
# Select top classifiers from baseline ranking
# -------------------------------------------------
TOP_K = 3
top_classifiers = [
    name for name in df_baseline_results["classifier"].head(TOP_K)
    if name in param_grids
]

print("Selected for tuning:")
for name in top_classifiers:
    print(f" - {name}")
print()

total_models = len(top_classifiers)

# -------------------------------------------------
# Perform grid search tuning for each selected classifier
# -------------------------------------------------
for i, name in enumerate(top_classifiers, 1):
    print(f"[{i}/{total_models}] Tuning: {name}")

    clf = candidate_classifiers[name]
    param_grid = param_grids[name]

    # -------------------------------------------------
    # Compute total number of parameter combinations
    # -------------------------------------------------
    n_combinations = len(list(product(*param_grid.values())))

    if VERBOSE:
        print(f"Parameter combinations: {n_combinations}")

    start_time = time.time()

    # -------------------------------------------------
    # Initialize GridSearchCV
    # -------------------------------------------------
    grid_search = GridSearchCV(
        estimator=clf,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv_strategy,
        verbose=1 if VERBOSE else 0,
        n_jobs=-1
    )

    # -------------------------------------------------
    # Fit grid search
    # -------------------------------------------------
    grid_search.fit(X_train, y_train_encoded)

    elapsed = time.time() - start_time

    # -------------------------------------------------
    # Extract best model and results
    # -------------------------------------------------
    best_model = grid_search.best_estimator_
    best_score = grid_search.best_score_

    print(f"Best ROC-AUC: {best_score:.4f}")
    print(f"Best params: {grid_search.best_params_}")
    print(f"Time: {elapsed:.2f} sec\n")

    # -------------------------------------------------
    # Store tuning results
    # -------------------------------------------------
    tuned_results.append({
        "classifier": name,
        "best_roc_auc": best_score,
        "best_params": grid_search.best_params_,
        "fit_time_sec": elapsed
    })

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("Tuning complete.")



Tuning top classifiers...

Selected for tuning:
 - RBF SVM
 - MLP
 - Random Forest

[1/3] Tuning: RBF SVM
Parameter combinations: 9
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best ROC-AUC: 0.8788
Best params: {'C': 10, 'gamma': 'scale'}
Time: 547.73 sec

[2/3] Tuning: MLP
Parameter combinations: 8
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Best ROC-AUC: 0.8674
Best params: {'alpha': 0.0001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.0005}
Time: 222.90 sec

[3/3] Tuning: Random Forest
Parameter combinations: 6
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best ROC-AUC: 0.8493
Best params: {'max_depth': 20, 'n_estimators': 200}
Time: 75.29 sec

Tuning complete.


### 🔷 Step 9 — Save Results

- Convert tuned classifier results into a DataFrame
- Validate that tuning results are available
- Sort tuned classifiers by best ROC AUC score
- Save baseline and tuned comparison results to CSV files
- Select top-performing tuned classifiers
- Export top classifier configurations to JSON format
- Display summary of top classifiers and their performance
- Confirm that all results have been saved successfully

---

In [9]:
# ============================================
# Step 9: Save Results
# ============================================

print("Saving classifier comparison results...\n")

# -------------------------------------------------
# Convert tuned results to DataFrame
# -------------------------------------------------
df_tuned_results = pd.DataFrame(tuned_results)

# -------------------------------------------------
# Validate tuned results are available
# -------------------------------------------------
if df_tuned_results.empty:
    raise ValueError("No tuned results found. Cell 8 may not have run successfully.")

# -------------------------------------------------
# Sort tuned results by best ROC-AUC
# -------------------------------------------------
df_tuned_results = df_tuned_results.sort_values(
    by="best_roc_auc",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Save baseline and tuned comparison tables
# -------------------------------------------------
df_baseline_results.to_csv(BASELINE_RESULTS_CSV, index=False)
df_tuned_results.to_csv(TUNED_RESULTS_CSV, index=False)

print(f"Saved baseline results: {BASELINE_RESULTS_CSV}")
print(f"Saved tuned results:    {TUNED_RESULTS_CSV}")

# -------------------------------------------------
# Select top-performing tuned classifiers
# -------------------------------------------------
TOP_K_FINAL = 2
top_models = df_tuned_results.head(TOP_K_FINAL)

# -------------------------------------------------
# Build summary for JSON export
# -------------------------------------------------
top_classifiers_summary = []

for _, row in top_models.iterrows():
    top_classifiers_summary.append({
        "classifier": str(row["classifier"]),
        "roc_auc": float(row["best_roc_auc"]),
        "params": dict(row["best_params"])
    })

# -------------------------------------------------
# Save best classifier configurations to JSON
# -------------------------------------------------
with open(BEST_CLASSIFIER_JSON, "w") as f:
    json.dump(top_classifiers_summary, f, indent=4)

print(f"Saved top {TOP_K_FINAL} classifier summaries: {BEST_CLASSIFIER_JSON}")

# -------------------------------------------------
# Display summary of saved results
# -------------------------------------------------
print(f"\nTop {TOP_K_FINAL} tuned classifiers:")
for i, model in enumerate(top_classifiers_summary, 1):
    print(f"{i}. {model['classifier']}")
    print(f"   ROC-AUC: {model['roc_auc']:.4f}")

    if VERBOSE:
        print(f"   Params:  {model['params']}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nSave complete.")



Saving classifier comparison results...

Saved baseline results: /content/dip-ai-image-detection/metadata/results/classifier_comparison_baseline.csv
Saved tuned results:    /content/dip-ai-image-detection/metadata/results/classifier_comparison_tuned.csv
Saved top 2 classifier summaries: /content/dip-ai-image-detection/metadata/results/best_model_config.json

Top 2 tuned classifiers:
1. RBF SVM
   ROC-AUC: 0.8788
   Params:  {'C': 10, 'gamma': 'scale'}
2. MLP
   ROC-AUC: 0.8674
   Params:  {'alpha': 0.0001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.0005}

Save complete.


### 🔷 Step 10 — Summary and Recommendations

- Define helper function for wrapped console output
- Verify that baseline and tuned results are available
- Identify the best baseline classifier
- Identify the top tuned classifiers
- Recommend top classifiers for final training and independent test evaluation
- Summarize the model selection procedure
- Confirm that the held-out test set was not used during classifier selection
- Optionally display additional interpretation when `VERBOSE=True`
- Confirm classifier selection notebook is complete

---

In [10]:
# ============================================
# Step 10: Summary and Recommendations
# ============================================

print("Summarizing classifier selection results...\n")

# -------------------------------------------------
# Import text wrapping utility
# -------------------------------------------------
import textwrap

# -------------------------------------------------
# Define wrapped print helper
# -------------------------------------------------
WRAP_WIDTH = 88

def wrapped_print(text="", indent=""):
    wrapped = textwrap.fill(
        str(text),
        width=WRAP_WIDTH,
        initial_indent=indent,
        subsequent_indent=indent
    )
    print(wrapped)

# -------------------------------------------------
# Validate required results are available
# -------------------------------------------------
if df_baseline_results.empty or df_tuned_results.empty:
    raise ValueError("Missing results. Ensure Steps 7 and 9 executed successfully.")

# -------------------------------------------------
# Identify best baseline classifier
# -------------------------------------------------
best_baseline_row = df_baseline_results.iloc[0]

print("Best baseline classifier:")
print(f"  Classifier: {best_baseline_row['classifier']}")
print(f"  ROC-AUC:    {best_baseline_row['roc_auc_mean']:.4f}")
print(f"  F1 Score:   {best_baseline_row['f1_mean']:.4f}")
print(f"  Accuracy:   {best_baseline_row['accuracy_mean']:.4f}")

# -------------------------------------------------
# Identify top tuned classifiers
# -------------------------------------------------
TOP_K_FINAL = 2
top_tuned_rows = df_tuned_results.head(TOP_K_FINAL)

print(f"\nTop {TOP_K_FINAL} tuned classifiers:")
for i, (_, row) in enumerate(top_tuned_rows.iterrows(), 1):
    print(f"  {i}. Classifier: {row['classifier']}")
    print(f"     ROC-AUC:    {row['best_roc_auc']:.4f}")

    if VERBOSE:
        wrapped_print(f"Parameters: {row['best_params']}", indent="     ")

# -------------------------------------------------
# Recommend classifiers for final evaluation
# -------------------------------------------------
recommended_classifiers = top_tuned_rows["classifier"].tolist()

print("\nRecommendation:")
wrapped_print(
    "The following classifiers are recommended for subsequent final training "
    "and independent test evaluation:",
    indent="  "
)

for clf in recommended_classifiers:
    print(f"   - {clf}")

# -------------------------------------------------
# Interpret selection procedure
# -------------------------------------------------
print("\nInterpretation:")
wrapped_print(
    "Classifier selection was performed using the normalized 26-feature DIP "
    "representation and stratified 5-fold cross-validation on the training "
    "set only. The held-out test set was not used in this notebook and "
    "remains reserved for final independent evaluation.",
    indent="  "
)

# -------------------------------------------------
# Optional verbose interpretation
# -------------------------------------------------
if VERBOSE:
    wrapped_print(
        "The baseline comparison identified the strongest classifier candidates, "
        "and light hyperparameter tuning was then applied to the top-performing "
        "models. The top two tuned classifiers are carried forward for final "
        "training and independent test evaluation in later notebooks.",
        indent="  "
    )

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nClassifier selection notebook complete.")



Summarizing classifier selection results...

Best baseline classifier:
  Classifier: RBF SVM
  ROC-AUC:    0.8608
  F1 Score:   0.7802
  Accuracy:   0.7779

Top 2 tuned classifiers:
  1. Classifier: RBF SVM
     ROC-AUC:    0.8788
     Parameters: {'C': 10, 'gamma': 'scale'}
  2. Classifier: MLP
     ROC-AUC:    0.8674
     Parameters: {'alpha': 0.0001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init':
     0.0005}

Recommendation:
  The following classifiers are recommended for subsequent final training and
  independent test evaluation:
   - RBF SVM
   - MLP

Interpretation:
  Classifier selection was performed using the normalized 26-feature DIP representation
  and stratified 5-fold cross-validation on the training set only. The held-out test set
  was not used in this notebook and remains reserved for final independent evaluation.
  The baseline comparison identified the strongest classifier candidates, and light
  hyperparameter tuning was then applied to the top-performing m